In [1]:
from utility.adjust_redshift import adjust_redshift

from sparcl.client import SparclClient

import numpy as np
import pandas as pd
import json
import h5py
import time

property_file_path = "data/target_properties.csv"
spectra_hdf5_path = "data/batches/spectra.h5"

In [2]:
df = pd.read_csv(property_file_path,index_col=0)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 404142 entries, 0 to 404141
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   targetid            404142 non-null  int64  
 1   z                   404142 non-null  float64
 2   zerr                404142 non-null  float64
 3   civ_1549_flux       404142 non-null  float64
 4   civ_1549_flux_ivar  404142 non-null  float64
 5   civ_1549_sigma      404142 non-null  float64
 6   flux_w3             404142 non-null  float64
 7   flux_ivar_w3        403793 non-null  float64
 8   desiname            404142 non-null  object 
dtypes: float64(7), int64(1), object(1)
memory usage: 30.8+ MB


In [3]:
# store the duplicate targets
duplicate_targetidxs = []

In [4]:
# detect duplicates, add to list, then remove from dataframe
duplicates = df[df["targetid"].duplicated()]["targetid"].unique()
# duplicate_targetidxs.append(duplicates)  # already done
df = df.drop_duplicates(subset="targetid",keep=False)

In [5]:
target_idxs = [int(x) for x in df["targetid"]]

n = len(target_idxs)
batch_size = 500 # sparcl cannot take more than 500 idxs at once 
n_batches = int((n / batch_size)+1)

print(f"{batch_size=}, {n_batches=}")

batch_size=500, n_batches=764


In [6]:
client = SparclClient()

announcement=Data set deprecation notice: on November 19, 2025 the SDSS/BOSS DR16 data sets were deprecated. Please use the new SDSS/BOSS DR17 data sets instead.


In [7]:
output_fields = ['targetid', 'redshift', 'wavelength', 'flux', 'ivar']

# results = client.retrieve_by_specid(specid_list=test_idxs, include=output_fields, dataset_list=["DESI-DR1"],limit=None,fmt="pandas")

In [ ]:
# duplicates = results[results["targetid"].duplicated()]["targetid"].unique()
# duplicate_targetidxs.append(duplicates)
# results = results.drop_duplicates(subset="targetid",keep=False)

In [ ]:
# for idx in results.index:
#     lam, flux, ivar = adjust_redshift(results.loc[idx,"redshift"], results.loc[idx,"wavelength"], results.loc[idx,"flux"], results.loc[idx,"ivar"])

#     valid_pixel_mask = ((lam > 1150) & (lam < 1810)) & (ivar > 0) & np.isfinite(ivar) & (flux != 0)

#     arr = np.array([lam[valid_pixel_mask],flux[valid_pixel_mask], ivar[valid_pixel_mask]])

#     with h5py.File("spectra.h5", "a") as f: 
#         dset = f.create_dataset(  
#             f"{results.loc[idx,'targetid']}", data=arr  
#         )



In [8]:
for i in range(530,630):
    tic = time.time()
    print(f"Start batch {i}")

    # get next 500 targetids
    curr_idxs = target_idxs[batch_size*i:batch_size*(i+1)]

    # retrieve spectra
    results = client.retrieve_by_specid(specid_list=curr_idxs, include=output_fields, dataset_list=["DESI-DR1"],limit=None,fmt="pandas")

    # detect duplicates, add to duplicate list, then remove them
    duplicates = results[results["targetid"].duplicated()]["targetid"].unique()
    duplicate_targetidxs.extend(duplicates)
    results = results.drop_duplicates(subset="targetid",keep=False)

    # redshift correct and truncate each spectrum to relevant window, then save as (3,N) numpy array to HDF5 file
    for idx in results.index:
        lam, flux, ivar = adjust_redshift(results.loc[idx,"redshift"], results.loc[idx,"wavelength"], results.loc[idx,"flux"], results.loc[idx,"ivar"])

        # valid pixels are: lambda inside relevant range (outer continuum window borders), flux is nonzero, ivar is nonzero and finite
        # get rid of pixels with 0 and infinite ivar (PyQSOFit does that similarly), as well as zero flux pixels (both PyQSOFit and VANDELS paper by Saldana-Lopez also does that)
        valid_pixel_mask = ((lam > 1150) & (lam < 1810)) & (ivar > 0) & np.isfinite(ivar) & (flux != 0)

        arr = np.array([lam[valid_pixel_mask],flux[valid_pixel_mask], ivar[valid_pixel_mask]])

        with h5py.File(spectra_hdf5_path, "a") as f: 
            dset = f.create_dataset(  
                f"{results.loc[idx,'targetid']}", data=arr  
            )
    
    toc = time.time() - tic
    print(f"Batch {i} finished in {toc:.2f}s\n")


Start batch 530
Batch 530 finished in 22.00s

Start batch 531
Batch 531 finished in 23.94s

Start batch 532
Batch 532 finished in 28.22s

Start batch 533
Batch 533 finished in 23.74s

Start batch 534
Batch 534 finished in 24.43s

Start batch 535
Batch 535 finished in 21.40s

Start batch 536
Batch 536 finished in 25.92s

Start batch 537
Batch 537 finished in 22.97s

Start batch 538
Batch 538 finished in 26.99s

Start batch 539
Batch 539 finished in 20.69s

Start batch 540
Batch 540 finished in 22.34s

Start batch 541
Batch 541 finished in 28.19s

Start batch 542
Batch 542 finished in 21.29s

Start batch 543
Batch 543 finished in 26.72s

Start batch 544
Batch 544 finished in 105.00s

Start batch 545
Batch 545 finished in 44.70s

Start batch 546
Batch 546 finished in 29.34s

Start batch 547
Batch 547 finished in 28.52s

Start batch 548
Batch 548 finished in 24.89s

Start batch 549
Batch 549 finished in 24.84s

Start batch 550
Batch 550 finished in 29.24s

Start batch 551
Batch 551 finishe

In [9]:
with h5py.File(spectra_hdf5_path, "r") as f:
    # test = f[f"{results.loc[400,'targetid']}"][()]
    print(len(set(f.keys())))

314062


In [11]:
with open("data/batches/duplicate_targets.txt","a") as f:
    f.write(str(duplicate_targetidxs))

In [10]:
duplicate_targetidxs

[39627884872077810,
 39627902769170686,
 39628001536640656,
 39632931265711564,
 39628527997289120,
 39628517855464495,
 39627767251208347,
 39628438381791144,
 39628517872240702,
 39628335856226710,
 39633103546748050,
 39632946537169791,
 39628319150315124,
 39632956091796685,
 39632951150903497,
 39633362792483578,
 39627751820365629,
 39627752139130769,
 39632931177629057,
 39633267653086458,
 39632936110130815,
 39628180222379151,
 39627709529200061,
 39632946411341733,
 39632955953381556,
 39628335839450328,
 39628395486642783,
 39628527695303086,
 39632930829503928,
 39627583343560930,
 39627800386212514,
 39633118667212252,
 39627818010673196,
 39632939935337389,
 39628522557279339,
 39627806459562124,
 39628176418149204,
 39628517155013785,
 39628516316153796,
 39628346962744325,
 39632956511227461,
 39627676662632224,
 39627933014296508,
 39633304554572149,
 39628449031128151,
 39627791406208732,
 39627842765456989,
 39628496728755013,
 39627775635622148,
 39628516316154957,
